Agora chegamos a parte dos testes de modelos para escolher qual utilizar apartir das métricas estatísticas.

## 03 - Model Experimentation

Com o dataset já tratado e pré-processado na etapa anterior, esta etapa tem como 
objetivo instanciar e treinar diferentes algoritmos de classificação, comparando 
seu desempenho para decidir qual utilizar no projeto final.

Modelos a serem testados:
- Regressão Logística
- Árvore de Decisão
- Random Forest
- KNN
- SVM

Dado o desbalanceamento identificado na etapa 2 (~73% não-churn / 27% churn), a 
avaliação priorizará métricas como F1-score, precision/recall e matriz de confusão, 
em vez de acurácia isolada.

In [3]:
# fazendo o load dos dados dentro da nossa pasta data_processed
# visando não alterar o dataset original.
import joblib  

dados_processados = joblib.load('data_processed/02_data_preparation.pkl')

X_train_raw = dados_processados['X_train_raw']
X_test_raw = dados_processados['X_test_raw']
X_train = dados_processados['X_train']
X_test = dados_processados['X_test']
y_train = dados_processados['y_train']
y_test = dados_processados['y_test']
pre_processador = dados_processados['preprocessador']
nomes_colunas = dados_processados['nomes_colunas']

Verificando se os dados foram carregados.

In [ ]:
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(4922, 30) (2110, 30)
(4922,) (2110,)


In [22]:
# Importando bibliotecas necessárias para testar os modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import pandas as pd # Para mostrar os resultados no final do comparativo.

**O fluxo geral que vamos seguir para cada um dos 5 modelos (mesma sequência, repetida):**

1 - **Instanciar** — criar o objeto do modelo (ex: `modelo = LogisticRegression(random_state=47)`), ainda sem nenhum aprendizado acontecendo.

2 - **Treinar** `(fit)` — `modelo.fit(X_train, y_train)`, onde o modelo efetivamente aprende os padrões dos dados de treino.

3 - **Prever** `(predict)` — `y_pred` = `modelo.predict(X_test)`, gerando as previsões do modelo para dados que ele nunca viu.

4 - **Avaliar** — comparar `y_pred` com o `y_test` real, usando métricas como F1-score, precision/recall e matriz de confusão (que já decidimos priorizar, por causa do desbalanceamento).

In [14]:
#Guardando os resultados para comparação no final
results = []

In [15]:
#LogisticRegression
reg_logistic = LogisticRegression(random_state=47)
reg_logistic.fit(X_train, y_train)
y_pred = reg_logistic.predict(X_test)

report = classification_report(y_test, y_pred, output_dict=True)
results.append({
    'modelo': 'Regressão Logística',
    'accuracy': report['accuracy'],
    'precision_yes': report['Yes']['precision'],
    'recall_yes': report['Yes']['recall'],
    'f1_yes': report['Yes']['f1-score']
})

Agora para fazermos a **comparação** precisamos importar uma biblioteca própria já pronta para que ela compare os nossos modelos, a biblioteca para isso é a `from sklearn.metrics import classification_report, confusion_matrix`

`classification_report(y_test, y_pred)` — gera um resumo completo de uma vez: precision, recall e F1-score, calculados separadamente para cada classe (Yes e No), além de uma média geral. É como um "boletim" do modelo.

`confusion_matrix(y_test, y_pred)` — mostra uma tabela 2x2 com os acertos e erros: quantos "No" o modelo previu certo, quantos "Yes" ele confundiu com "No" (e vice-versa). Essa matriz é a base de onde vêm precision e recall, então ajuda a entender os números do relatório, não só ler eles.

#### Lembrando que decidimos priorizar **F1-score, precision/recall e matriz de confusão** (por causa do desbalanceamento).


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Verificando se o modelo foi treinado corretamente.
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred), "Matriz de confusão")

              precision    recall  f1-score   support

          No       0.85      0.89      0.87      1549
         Yes       0.65      0.56      0.60       561

    accuracy                           0.80      2110
   macro avg       0.75      0.72      0.73      2110
weighted avg       0.79      0.80      0.80      2110

[[1376  173]
 [ 246  315]] Matriz de confusão


Pesquisando e analisando código disponibilizado pelo meu professor da faculdade, percebi que é possível gerar um loop para treinar os modelos de uma só vez, porém vou manter a regressão logistica fora do loop para que sejá possível lembrar a lógica por trás do treino.

In [18]:
#Utilizando laço for para criar um loop e treinar todos modelos restantes
modelos = {
    'Árvore de Decisão': DecisionTreeClassifier(random_state=47),
    'Random Forest': RandomForestClassifier(random_state=47),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(random_state=47)
}

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    results.append({
        'modelo': nome,
        'accuracy': report['accuracy'],
        'precision_yes': report['Yes']['precision'],
        'recall_yes': report['Yes']['recall'],
        'f1_yes': report['Yes']['f1-score']
    })

In [25]:
# Mostrando os resultados dos comparativos
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='f1_yes', ascending=False)
df_results

,modelo,accuracy,precision_yes,recall_yes,f1_yes
0,Regressão Logística,0.801422,0.645492,0.561497,0.600572
3,KNN,0.762559,0.557915,0.515152,0.535681
2,Random Forest,0.780095,0.613583,0.467023,0.530364
4,SVM,0.789573,0.656000,0.438503,0.525641
1,Árvore de Decisão,0.724171,0.482051,0.502674,0.492147


### Por que o F1-score da classe `Yes` como métrica principal?

Dado o desbalanceamento identificado na etapa 2 (~73% não-churn / 27% churn), a acurácia 
isolada não é confiável para comparar os modelos: um modelo que sempre previsse "No" já 
acertaria ~73% das vezes, sem identificar corretamente nenhum cliente propenso a cancelar.

Entre as métricas focadas na classe `Yes` (a que realmente importa para o negócio, já que 
o objetivo é identificar clientes em risco de churn para ação de retenção), o F1-score foi 
escolhido como critério principal de ranqueamento por representar o **equilíbrio** entre:

- **Precision (Yes)**: das vezes que o modelo previu churn, quantas estavam corretas — 
  importante para não gerar ações de retenção desnecessárias em clientes que não iam cancelar.
- **Recall (Yes)**: dos clientes que realmente deram churn, quantos o modelo conseguiu 
  identificar — importante para não deixar passar clientes que poderiam ser retidos.

Usar apenas uma das duas isoladamente favoreceria modelos "extremos" (ex: um modelo com 
recall altíssimo mas precision baixa, gerando muitos falsos alarmes). O F1-score, sendo a 
média harmônica entre as duas, penaliza esse desequilíbrio e serve como um critério de 
comparação mais equilibrado para esta análise preliminar.

**Resultado**: a **Regressão Logística** obteve o maior F1-score da classe `Yes` (0.60) 
entre os 5 modelos testados, com o melhor recall (0.56) do grupo — sendo o modelo mais 
consistente para seguir como candidato ao projeto final, considerando o objetivo de 
identificar clientes propensos ao churn.